# framediff: animations rebuilt frame by frame

`Decomposer().explain_animation(before, frames, before_name, frame_name)` writes one block that rebuilds a move's animation into a dict `anim` keyed by frame index, one piece per frame, and ends in one assert over the whole dict. The board `g` holds the newest built frame throughout, so a frame built from the one before it needs no line to start; any other source starts with a copy of it.

A frame that repeats an earlier built frame exactly is a copy of it, found before anything is searched. Any other frame is built by the cheapest script among the three built frames before it (most recent first) and the frame before the move; the order only breaks ties. The budget is per frame (`Decomposer.animation_max_bits`): a frame no candidate fits is taken as it is, and the frames after it still build on it.

Without a stride, the block is tried with every frame, then every 2nd, 3rd, 4th and 6th, until it fits the character limit (`framediff.animation.MAX_CHARS`); an attempt is abandoned the moment it passes the limit, and the last stride is written whole.

The animations come from a recorded run (`run_animations.py`), as the seal's kernel keeps them: consecutive repeats dropped, trailing frames equal to the settled frame dropped, and the move that wins a level left out. The names are the ones the model would read, `history[i]` counted within the level. For every game, its first animation and its longest.

In [ ]:
import sys
import time

sys.path.insert(0, ".")

from IPython.display import HTML, display
from run_animations import DEFAULT_RUN, first_and_longest, load_states

from framediff import Decomposer
from framediff.diagnostics import show_animation

RUN = DEFAULT_RUN
states = load_states(RUN)
chosen = first_and_longest(states)
len(states), len(chosen)

In [ ]:
explained = []
for game_id, pair in chosen.items():
    for which, animation in zip(("first", "longest"), pair):
        if which == "longest" and animation is pair[0]:
            continue
        started = time.perf_counter()
        found = Decomposer().explain_animation(
            animation.before, animation.frames, animation.before_name, animation.frame_name
        )
        explained.append((which, animation, found, time.perf_counter() - started))
len(explained)

## Summary

One row per animation shown below: frames kept (and sent), the stride the block settled on to stay within the character limit, how many of the built frames are scripts and what they cost, how many are taken as they are, and the block's lines, characters and search time. A game whose first animation is also its longest has one row.


In [ ]:
header = [
    "game",
    "which",
    "level",
    "turn",
    "action",
    "frames",
    "sent",
    "stride",
    "built",
    "bits",
    "max bits",
    "as is",
    "lines",
    "chars",
    "s",
]
rows = []
for which, animation, found, seconds in explained:
    built = [step.cost for step in found.frames if step.cost is not None]
    rows.append(
        [
            animation.game_id,
            which,
            animation.level,
            f"history[{animation.turn}]",
            animation.action,
            len(animation.frames),
            animation.raw_length,
            found.stride,
            len(built),
            f"{sum(built):g}",
            f"{max(built, default=0):g}",
            len(found.frames) - len(built),
            len(found.code.splitlines()),
            len(found.code),
            f"{seconds:.1f}",
        ]
    )
cell = "padding: 2px 10px 2px 0; text-align: right; font: 12px monospace"
display(
    HTML(
        '<table style="border-collapse: collapse">'
        + "<tr>"
        + "".join(f'<th style="{cell}">{name}</th>' for name in header)
        + "</tr>"
        + "".join("<tr>" + "".join(f'<td style="{cell}">{value}</td>' for value in row) + "</tr>" for row in rows)
        + "</table>"
    )
)

## Every game: its first animation, then its longest

Each panel: the frame before the move, each animation frame captioned with its source and cost (cells that differ from the source outlined), the settled frame, and the block. Long animations show their first frames only; the block is always whole.

In [ ]:
for which, animation, found, seconds in explained:
    display(
        show_animation(
            animation.before,
            animation.frames,
            found,
            settled=animation.settled,
            title=f"{which}: {animation.title}",
            seconds=seconds,
        )
    )